# 05 · Validation plan — Layer 4 hook, packaging & cohort adoption

**Standard slot:** *validation plan.* **For Project 05 this means:** this project ships *software*, so
the "plan" is how the engine gets **adopted and maintained**: wire the optional **Layer 4 (short MD)**
hook, sketch a pip-installable package (stretch), and document **how the cohort proposes cutoff
changes via pull request** — never by silently overwriting the shared file (D4/D5).

Run `00`–`04` first.

## Setup paths

In [ ]:
import sys, os
# Make the project's scripts/ and the cohort's shared/ importable.
# Adjust these if your Colab working directory differs (see 00_setup §5 for Drive mounting).
sys.path.insert(0, os.path.abspath("../scripts"))
sys.path.insert(0, os.path.abspath("../../../shared"))
os.makedirs("results", exist_ok=True)
print("paths ready; cwd =", os.getcwd())

## 1 · Optional **Layer 4 — dynamics (short MD)** hook

Static metrics can pass a design that *folds but melts*. Layer 4 runs a short MD and checks the
structure does not drift (`md_rmsd ≤ max_md_rmsd`). It is **expensive and optional** — at campaign
scale it is usually not worth it; reserve it for the final short-list. The planted
`BAD_L4_dynamics` design passes L1–L3 and only L4 catches it.

In [ ]:
import filtering_pipeline as fp
import make_example_pool as mep

planted = {r["design_id"]: r for r in mep.known_designs()}
def design_from_row(row):
    drop = {"truth", "note", "design_id"}
    return fp.Design(design_id=row["design_id"], sequence="M",
                     **{k: v for k, v in row.items() if k not in drop})

cut = fp.DEFAULT_CUTOFFS["monomer"]
melt = design_from_row(planted["EXAMPLE_DATA_BAD_L4_dynamics"])
assert fp.self_consistency(melt, cut) is True, "folds-but-melts passes L1"
assert fp.physics_filter(melt, cut) is True, "...and L3"
assert fp.dynamics_filter(melt) is False, "...but L4 (MD drift) catches it"
print(f"BAD_L4_dynamics: md_rmsd={melt.md_rmsd}  -> L1/L3 pass, L4 fails  notes={melt.notes}")
print("Layer 4 verified. (Run the full pipeline with use_layers=(1,2,3,4) to include MD.)")

### Where the MD numbers come from (behind the boundary)

The engine only *thresholds* `md_rmsd`. Producing it is an upstream OpenMM job — solvate, minimize,
short NVT/NPT run, measure mean backbone RMSD vs the design. That is GPU/CPU time you spend only on a
short-list. Pseudocode (implement on Colab with OpenMM; pin the version):

In [ ]:
# Pseudocode — produce md_rmsd upstream, then set it on the Design (engine stays GPU-free):
#
#   from openmm.app import *; from openmm import *; from openmm.unit import *
#   # 1. load design PDB, add H, solvate, add ions
#   # 2. minimize; equilibrate; run 10-50 ns NVT/NPT
#   # 3. align each frame to frame 0; md_rmsd = mean backbone RMSD over the trajectory
#   design.md_rmsd = computed_md_rmsd
#   fp.dynamics_filter(design, max_md_rmsd=3.0)
print("MD is the optional, expensive layer — only worth it on the final short-list.")

## 2 · Package as a pip-installable module `[stretch]`

To make the engine trivially adoptable, it can ship as a tiny package. Below is a **minimal
`pyproject.toml` scaffold shown in a cell** — do **not** commit it as a real package in this repo; the
canonical home of the engine is `shared/filtering_pipeline.py`. This is just to show the shape.

In [ ]:
pyproject = """
[build-system]
requires = ["setuptools>=68", "wheel"]
build-backend = "setuptools.build_meta"

[project]
name = "denovo-filtering-pipeline"
version = "1.0.0"
description = "4-layer in-silico triage filter for de novo protein design (cohort shared engine)."
requires-python = ">=3.9"
dependencies = ["numpy", "pandas", "matplotlib", "biopython>=1.84"]

[project.optional-dependencies]
test = ["pytest"]
"""
print(pyproject)
print("# Scaffold only — the engine's canonical home is shared/filtering_pipeline.py.")
print("# If packaged: `pip install -e .` then `import filtering_pipeline`. Do NOT commit this here.")

## 3 · How the cohort adopts cutoff changes (via PR, not silent overwrite)

Your calibration may justify changing `DEFAULT_CUTOFFS`. Because every project depends on this engine,
changes go through a **reviewed pull request** with evidence — never a notebook silently overwriting
the shared file. Below shows the current cutoffs and a *template* for proposing new ones; put the
proposal (with enrichment + N) in the PR description.

In [ ]:
import filtering_pipeline as fp
print("CURRENT shared DEFAULT_CUTOFFS:")
for k, v in fp.DEFAULT_CUTOFFS.items():
    print(" ", k, v)

# Proposed change — fill from YOUR notebook-04 enrichment study, then open a PR. Example shape:
proposed = {
    # "monomer": dict(scrmsd=1.8, plddt=88, pae=None),   # evidence: enrichment X->Y at N=...
}
print("\nProposed (put these + the enrichment/N evidence in your PR description):")
print(proposed)
print("\nDo NOT write this back to shared/filtering_pipeline.py from here — open a reviewed PR.")

## 4 · The cohort-adoption guide (what downstream projects need)

Write a short "how to use this engine" page for Projects 06–25:
- **Import:** `import filtering_pipeline as fp` (after the Setup-paths cell).
- **Build:** one `fp.Design` per candidate; set the metrics your predictors produced.
- **Run:** `fp.run_pipeline(designs, design_type=..., use_layers=(1,2,3))`.
- **Report:** `fp.report(df, save_prefix="results/<proj>")` → ranked CSV + survival figure.
- **Controls every project must include:** a known-good positive control + a negative control
  (scrambled interface / catalytic dead-mutant) in their pool, so the funnel is interpretable.
- **The honest caveat to repeat in every report:** the filter *enriches*, it does not *guarantee*;
  report false positives among survivors and your hit rate, not the cherry.

## D4 / D5 checklist
- [ ] Optional Layer 4 (short MD) hook verified on the planted folds-but-melts design.
- [ ] `pyproject.toml` packaging scaffold shown (not committed as a real package).
- [ ] Cohort-adoption guide written; cutoff changes routed through reviewed PRs (no silent overwrite).
- [ ] Thesis chapter + 15-min talk + `v1.0` tagged release.
- [ ] Engine landed as `shared/filtering_pipeline.py` — Projects 01–25 now stand on it.

You're done — and the cohort now has a tested, documented triage engine.